In [14]:
import json
import re

def all_fields_match(metadata, fields):
    """
    takes a dictionary containing the mapped metadata in metasra and 
    a dictionary of field names to filter as key and corresponding filter terms
    as regular expressions and returns True if all fields match the filter term
    else False
    
    :param metadata:   dictionary containing the mapped metadata from metasra
    :param fields:     dictonary with fields to filter as key and values to filter as compiled re objects (see re package)
    
    :return:           True if all fields match else False
    """
    matching_results = []
    for field, matcher in fields.items():
        fieldvalue = metadata[field]
        match = matcher.match(fieldvalue)
        matching_results.append(True if match else False)
    
    return all(matching_results)
    

def filter_metasra_by_fields(db, fields):
    """
    takes a the loaded metasra db and a dictionary with fields to filter as keys and 
    field values to filter by as corresponding values.
    
    Returns the filtered db
    
    :param db:        metasra db dictionary returned by json.load
    :param fields:    dictionary with fields to filter as keys and values to filter by as values
    
    :return:          filtered db
    """
    
    fields = {
        field: re.compile(filtervalue) for field, filtervalue in fields.items()
    }
    
    filtered_db = dict()
    for sraid, mapped_metadata in db.items():
        if all_fields_match(mapped_metadata, fields):
            filtered_db[sraid] = mapped_metadata
        
    return filtered_db
    

fields = {
    'assay': 'RNA-seq',
    'species': 'human'
}


# metasra db is indexed by SRA sample accessions start with SRS
with open('metasra.v1-8.json', 'r') as metasradb:
    db = json.load(metasradb)
    filtered_db = filter_metasra_by_fields(
        db, 
        fields
    )

len(filtered_db)

343431

In [9]:
import urllib3
import certifi
import json
import pandas as pd
from io import StringIO


http = urllib3.PoolManager(
    ca_certs = certifi.where(),
    cert_reqs = 'CERT_NONE'                      
)

r = http.request(
    'GET', 
    'http://metasra.biostat.wisc.edu/api/v01/samples.csv',
    fields = {
        'study': 'SRP015668',
        'species': 'human', 
        'assay': 'RNA-seq', 
        'limit': 1,
        'skip': 0
    }
)

#json.loads(r.data.decode('utf-8'))
data = StringIO(r.data.decode('utf-8'))
pd.read_csv(data)

/users/daniel.malzl/.conda/envs/scllm/lib/python3.12/site-packages/urllib3/connectionpool.py:1100: InsecureRequestWarning: Unverified HTTPS request is being made to host 'metasra.biostat.wisc.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,study_id,study_title,sample_id,sample_name,sample_type,sample_type_confidence,mapped_ontology_ids,mapped_ontology_terms,raw_SRA_metadata,sample_species,assay
0,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361718,Brain cortex,tissue,0.935183,"EFO:0003534, UBERON:0003100, UBERON:0013540","dorsal telencephalon, female organism, Brodman...",disease state: Control; gender: female; tissue...,human,RNA-seq
1,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361724,Brain cortex,tissue,0.935183,"EFO:0003534, UBERON:0003100, UBERON:0013540","dorsal telencephalon, female organism, Brodman...",disease state: Control; gender: female; tissue...,human,RNA-seq
2,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361728,Brain cortex,tissue,0.935183,"EFO:0003534, UBERON:0003100, UBERON:0013540","dorsal telencephalon, female organism, Brodman...",disease state: Control; gender: female; tissue...,human,RNA-seq
3,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361723,Brain cortex,tissue,0.935183,"EFO:0003534, UBERON:0003100, UBERON:0013540","dorsal telencephalon, female organism, Brodman...",disease state: Control; gender: female; tissue...,human,RNA-seq
4,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361714,Brain cortex,tissue,0.935183,"EFO:0003534, UBERON:0003100, UBERON:0013540","dorsal telencephalon, female organism, Brodman...",disease state: Control; gender: female; tissue...,human,RNA-seq
5,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361716,Brain cortex,tissue,0.920950,"EFO:0003534, UBERON:0003101, UBERON:0013540","dorsal telencephalon, male organism, Brodmann ...",disease state: Control; gender: male; tissue: ...,human,RNA-seq
6,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361725,Brain cortex,tissue,0.920950,"EFO:0003534, UBERON:0003101, UBERON:0013540","dorsal telencephalon, male organism, Brodmann ...",disease state: Control; gender: male; tissue: ...,human,RNA-seq
7,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361719,Brain cortex,tissue,0.920950,"EFO:0003534, UBERON:0003101, UBERON:0013540","dorsal telencephalon, male organism, Brodmann ...",disease state: Control; gender: male; tissue: ...,human,RNA-seq
8,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361717,Brain cortex,tissue,0.920950,"EFO:0003534, UBERON:0003101, UBERON:0013540","dorsal telencephalon, male organism, Brodmann ...",disease state: Control; gender: male; tissue: ...,human,RNA-seq
9,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361715,Brain cortex,tissue,0.920950,"EFO:0003534, UBERON:0003101, UBERON:0013540","dorsal telencephalon, male organism, Brodmann ...",disease state: Control; gender: male; tissue: ...,human,RNA-seq


In [ ]:
from Bio import Entrez
Entrez.email = 'dmalzl@cemm.oeaw.ac.at'

# there seems to be no way of posting a list of accessions to the WebEnv
# so we have to do this one by one which might be extremely slow
uids = set()
for srs in filtered_db.keys():
    response_handle = Entrez.esearch(db = 'sra', term = srs)
    id_list = Entrez.read(response_handle)['IdList']
    uids.update(id_list)

In [32]:
response_handle = Entrez.epost(db = 'sra', id = ','.join(uids))
web_env_info = Entrez.read(response_handle)

{'QueryKey': '1', 'WebEnv': 'MCID_654f8bf9d080b56ba32dd77a'}

In [20]:
accession_matchers = {
    'experiment': re.compile('Experiment acc="([SRX0-9]+)'),
    'sample': re.compile('Sample acc="([SRS0-9]+)'),
    'study': = re.compile('Study acc="([SRP0-9]+)'),
    'biosample': re.compile('Biosample>([SAMN0-9]+)'),
    'bioproject': re.compile('Bioproject>([PRJNA0-9]+)')
}
# needs to be batched to retrieve everything correctly
response_handle = Entrez.esummary(
    db = 'sra',
    retstart = retstart,
    retmax = retmax,
    WebEnv = web_env_info['WebEnv'],
    query_key = web_env_info['QueryKey']
)

# this likely needs to go into a function later since we are iterating over multiple things here
# this needs a bit of rethinking to make it fit into a dataframe easily
response = Entrez.read(response_handle)
sra_accessions = dict()
for metadata in response:
    uid = metadata['Id']
    xml_metadata = metadata['ExpXml']
    accessions = dict()
    for accession_type, accession_matcher in accession_matchers.items():
        match_results = accession_matcher.search(xml_metadata)
        accession = match_results.groups()[0]
        accessions[accession_type] = accession
    
    sra_accessions[uid] = accessions

('SAMN01163627',)

In [2]:
from Bio import Entrez
Entrez.email = 'dmalzl@cemm.oeaw.ac.at'

r = Entrez.esummary(db = 'sra', id = '2398520')
Entrez.read(r)

[{'Item': [], 'Id': '2398520', 'ExpXml': '<Summary><Title>GSM1249878: Gro-seq-siBrd4-2-re2-HEK293T; Homo sapiens; RNA-Seq</Title><Platform instrument_model="Illumina HiSeq 2000">ILLUMINA</Platform><Statistics total_runs="1" total_spots="46963120" total_bases="2348156000" total_size="1571021146" load_done="true" cluster_name="public"/></Summary><Submitter acc="SRA108040" center_name="GEO" contact_name="Gene Expression Omnibus (GEO), NCBI, NLM, NIH, htt" lab_name=""/><Experiment acc="SRX1671899" ver="1" status="public" name="GSM1249878: Gro-seq-siBrd4-2-re2-HEK293T; Homo sapiens; RNA-Seq"/><Study acc="SRP031885" name="Brd4 and JMJD6-associated Anti-pause Enhancers in Regulation of Transcriptional Pause Release"/><Organism taxid="9606" ScientificName="Homo sapiens"/><Sample acc="SRS1369050" name=""/><Instrument ILLUMINA="Illumina HiSeq 2000"/><Library_descriptor><LIBRARY_STRATEGY>RNA-Seq</LIBRARY_STRATEGY><LIBRARY_SOURCE>TRANSCRIPTOMIC</LIBRARY_SOURCE><LIBRARY_SELECTION>cDNA</LIBRARY_SELE

In [5]:
response_handle = Entrez.esearch(db = 'sra', term = 'GSM1249878')
Entrez.read(response_handle)

{'Count': '1', 'RetMax': '1', 'RetStart': '0', 'IdList': ['2398520'], 'TranslationSet': [], 'TranslationStack': [{'Term': 'GSM1249878[All Fields]', 'Field': 'All Fields', 'Count': '1', 'Explode': 'N'}, 'GROUP'], 'QueryTranslation': 'GSM1249878[All Fields]'}

In [34]:
Entrez.read(r)

NotXMLError: Failed to parse the XML data (no element found: line 1, column 0). Please make sure that the input data are in XML format.